# Installation of required dependencies


# Financial Intelligence Engine (FIE)
===================================

Reproducible computational implementation of the operator-based
visual analytics framework

    F = Pi o Psi o C o G o P o D

The script generates a controlled synthetic financial experiment combining:

    D  : nonlinear latent risk representation,
    P  : probabilistic risk scoring,
    G  : weighted financial-network construction,
    C  : shortest-path-based contagion dynamics,
    Psi: mapping of quantitative states into visual attributes,
    Pi : perspective-like geometric projection.

The present implementation is a synthetic proof of concept. It is not an
empirically calibrated forecasting model and does not introduce a new
financial contagion mechanism. Its purpose is to provide a transparent,
modular, and reproducible computational realization of the operator-based
architecture described in the accompanying manuscript.

Reproducibility:
    - fixed NumPy random seed,
    - explicit network-generation mechanism,
    - stored simulation outputs,
    - deterministic rendering configuration conditional on the seed.

Author: Ana Isabel Castillo Pereda; e-mail: anacp20@gmail.com
Scientific visualization: @IsabelCasPe - Maths

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.animation import FuncAnimation, FFMpegWriter
from matplotlib.patches import Rectangle, FancyBboxPatch, Circle
import networkx as nx
from collections import defaultdict
import warnings

warnings.filterwarnings("ignore")

# ============================================================
# CONFIGURAÇÃO CIENTÍFICA
# ============================================================

In [ ]:
np.random.seed(2026)

FPS = 24
DURATION = 28
FRAMES = FPS * DURATION
DPI = 150
FIGSIZE = (16, 9)
OUT_VIDEO = "FIE_scientific_v1.mp4"
OUT_DATA  = "FIE_results.npz"

# Parâmetros do modelo (fáceis de variar depois)
N_OBS       = 520
N_NODES     = 24
LAMBDA_DECAY = 0.42          # atenuação do contágio
SHOCK_START  = 0.40
SHOCK_DUR    = 0.38

# Cores
bg     = "#030712"
panel  = "#07152A"
panel2 = "#0B1F3A"
white  = "#FFFFFF"
'...'

plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["mathtext.fontset"] = "cm"

# ============================================================
# 1. DATA ENGINE - OPERATORS D AND P
# ============================================================

In [ ]:
class DataEngine:
    """
    Synthetic data and probabilistic-risk engine.

    Operator D maps two-dimensional synthetic observations into a nonlinear
    latent risk signal z_i = f(X1_i, X2_i). Operator P subsequently maps this
    latent signal into probabilistic scores through the logistic function.

    The time-dependent score path is generated by adding a decreasing Gaussian
    perturbation to the latent state. This is a controlled synthetic temporal
    mechanism, not a fitted forecasting process.
    """

    def __init__(self, n=N_OBS, frames=FRAMES):
        self.n = n
        self.frames = frames

        # Synthetic input space X: independent standard-normal features.
        self.x1 = np.random.normal(0, 1, n)
        self.x2 = np.random.normal(0, 1, n)

        # Operator D: nonlinear latent risk representation.
        # z_i = 1.25 sin(1.4 X1_i) + 0.95 cos(1.2 X2_i)
        #       + 0.55 X1_i X2_i - 0.25 X1_i^2
        # This stage generates continuous heterogeneous risk geometry.
        self.latent = (
            1.25 * np.sin(1.40 * self.x1) +
            0.95 * np.cos(1.20 * self.x2) +
            0.55 * self.x1 * self.x2 -
            0.25 * self.x1**2
        )

        # Operator P: logistic mapping p_i = sigma(z_i) in (0,1).
        self.p_true = 1.0 / (1.0 + np.exp(-self.latent))
        self.y = (np.random.rand(n) < self.p_true).astype(int)

        # Time-dependent synthetic probabilistic states.
        # Gaussian perturbations decay linearly over time; this is a visual-
        # analytical device rather than a learned training trajectory.
        self.score_path = np.zeros((frames, n))
        for f in range(frames):
            t = f / max(frames - 1, 1)
            noise = (1.0 - t) * np.random.normal(0, 0.52, n)
            self.score_path[f] = 1.0 / (1.0 + np.exp(-(self.latent + noise)))

        # Aggregate probabilistic state used by the dashboard.
        # This is descriptive of the simulation, not a predictive metric.
        self.mean_risk = np.mean(self.score_path, axis=1)

# ============================================================
# 2. NETWORK ENGINE - OPERATORS G AND C
# ============================================================

In [ ]:
class NetworkEngine:
    """
    Synthetic financial-network and contagion engine.

    Operator G constructs a weighted undirected graph with sector-dependent
    connectivity. Operator C propagates a normalized shock from the most
    vulnerable node according to

        C_i(t) = S(t) exp[-lambda d_G(v_i, v_0)].

    The contagion rule is deliberately simple and transparent; it is used to
    demonstrate the architecture rather than to replace established models.
    """

    def __init__(self, m=N_NODES, lambda_decay=LAMBDA_DECAY):
        self.m = m
        self.lambda_decay = lambda_decay

        # Geometric layout used only for visualization; it does not define
        # graph topology or contagion paths.
        theta = np.linspace(0, 2*np.pi, m, endpoint=False)
        radius = 0.34 + 0.05 * np.sin(3 * theta)
        self.x = 0.5 + radius * np.cos(theta)
        self.y = 0.5 + 0.80 * radius * np.sin(theta)
        self.z = 0.40 * np.sin(2 * theta) + 0.15 * np.random.normal(size=m)

        self.sector = np.random.choice([0, 1, 2], size=m)
        self.risk   = np.random.uniform(0.20, 0.98, m)

        # Operator G: stochastic weighted-network construction.
        # Same-sector nodes receive a higher connection probability.
        self.edges = []
        for i in range(m):
            for j in range(i + 1, m):
                p = 0.06 + 0.16 * (self.sector[i] == self.sector[j]) + 0.025 * np.random.rand()
                if np.random.rand() < p:
                    w = np.random.uniform(0.30, 1.0)
                    self.edges.append((i, j, w))

        # Explicit NetworkX graph used for topological calculations.
        # Edge weights are retained for downstream visual encoding.
        self.G = nx.Graph()
        self.G.add_nodes_from(range(m))
        for i, j, w in self.edges:
            self.G.add_edge(i, j, weight=w)

        # Shock source v_0: node with maximum assigned vulnerability.
        # Vulnerability and structural centrality remain distinct concepts.
        self.source = int(np.argmax(self.risk))

        # Shortest-path distances d_G(v_i,v_0) used by operator C.
        # Unreachable nodes receive a large sentinel distance.
        self.distances = nx.single_source_shortest_path_length(self.G, self.source)
        for i in range(m):
            if i not in self.distances:
                self.distances[i] = 999   # nó isolado

    def contagion_level(self, i, t):
        """Return C_i(t) = S(t) exp[-lambda d_i] for node i at normalized time t."""
        shock = np.clip((t - SHOCK_START) / SHOCK_DUR, 0.0, 1.0)
        dist  = self.distances[i]
        return shock * np.exp(-self.lambda_decay * dist)

    def compute_cascade_metrics(self, frames):
        """
        Pre-compute descriptive contagion indicators:
        cascade size, maximum contagion, and network-average contagion.
        These summarize the synthetic experiment and are not empirical market
        risk estimates.
        """
        cascade_size = np.zeros(frames)
        max_contagion = np.zeros(frames)
        avg_contagion = np.zeros(frames)

        for f in range(frames):
            t = f / max(frames - 1, 1)
            levels = np.array([self.contagion_level(i, t) for i in range(self.m)])
            cascade_size[f]  = np.sum(levels > 0.25)
            max_contagion[f] = np.max(levels)
            avg_contagion[f] = np.mean(levels)

        return cascade_size, max_contagion, avg_contagion

# ============================================================
# 3. VISUAL MAPPING AND PROJECTION - OPERATORS Psi AND Pi
# ============================================================

In [ ]:
class CameraEngine:
    """Perspective-like projection operator Pi; changes representation only."""
    def __init__(self, frames=FRAMES):
        self.frames = frames

    def state(self, frame):
        t = frame / max(self.frames - 1, 1)
        zoom = 1.0 + 0.13 * np.sin(2 * np.pi * t) + 0.16 * np.exp(-((t - 0.60) / 0.15)**2)
        rot  = 0.16 * np.sin(2 * np.pi * (t + 0.10))
        tilt = 0.09 * np.sin(2 * np.pi * (t + 0.28))
        return zoom, rot, tilt

    def project(self, x, y, z, frame):
        zoom, rot, tilt = self.state(frame)
        xc = x - 0.5
        yc = y - 0.5
        xr = np.cos(rot) * xc - np.sin(rot) * yc
        yr = np.sin(rot) * xc + np.cos(rot) * yc
        yr = yr + tilt * z
        depth = 1.0 / (1.0 + 0.60 * z)
        xp = 0.5 + zoom * xr * depth
        yp = 0.5 + zoom * yr * depth
        size_scale  = np.clip(0.75 + 0.50 * depth, 0.45, 1.70)
        alpha_scale = np.clip(0.40 + 0.45 * depth, 0.25, 1.0)
        return xp, yp, size_scale, alpha_scale


class ParticleEngine:
    """Decorative particle layer; carries no analytical information."""
    def __init__(self, p=180):
        self.p = p
        self.x = np.random.rand(p)
        self.y = np.random.rand(p)
        self.z = np.random.uniform(-0.5, 0.7, p)
        self.phase = np.random.uniform(0, 2 * np.pi, p)
        self.speed = np.random.uniform(0.004, 0.014, p)
        self.size  = np.random.uniform(3, 11, p)

    def draw(self, ax, frame, active):
        drift = frame * self.speed
        xx = (self.x + 0.016 * np.sin(drift + self.phase)) % 1
        yy = (self.y + 0.011 * np.cos(1.25 * drift + self.phase)) % 1
        alpha = 0.04 + 0.10 * (0.5 + 0.5 * np.sin(frame * 0.05 + self.phase))
        ax.scatter(xx, yy, s=self.size * (1.0 + 0.6 * (self.z + 0.5)),
                   color=active, alpha=alpha, transform=ax.transAxes, zorder=-7)


class BackgroundEngine:
    """Decorative scientific background; excluded from quantitative dynamics."""
    def __init__(self):
        self.x_wave = np.linspace(0, 1, 480)
        self.n_waves = 40
        self.offsets = np.linspace(-0.06, 1.06, self.n_waves)
        self.phase = np.random.uniform(0, 2 * np.pi, self.n_waves)
        self.amp   = np.random.uniform(0.009, 0.024, self.n_waves)
        self.speed = np.random.uniform(0.009, 0.040, self.n_waves)
        self.stars_x = np.random.rand(90)
        self.stars_y = np.random.rand(90)
        self.stars_s = np.random.uniform(2, 7, 90)
        self.stars_p = np.random.uniform(0, 2 * np.pi, 90)

    def draw(self, ax, frame, active, intensity):
        ax.clear()
        ax.set_facecolor(bg)
        ax.axis("off")
        for k, y0 in enumerate(self.offsets):
            yv = y0 + intensity * self.amp[k] * np.sin(
                2 * np.pi * (2.1 * self.x_wave + self.speed[k] * frame) + self.phase[k]
            )
            ax.plot(self.x_wave, yv, color=active, alpha=0.04 + 0.022 * intensity,
                    linewidth=1.1, transform=ax.transAxes, zorder=-10)
        star_alpha = 0.10 + 0.14 * (0.5 + 0.5 * np.sin(frame * 0.09 + self.stars_p))
        ax.scatter(self.stars_x, self.stars_y, s=self.stars_s, color=white,
                   alpha=star_alpha, transform=ax.transAxes, zorder=-9)


def style_panel(ax):
    ax.set_facecolor((0.03, 0.09, 0.18, 0.78))
    for s in ax.spines.values():
        s.set_edgecolor(gold)
        s.set_linewidth(0.80)
        s.set_alpha(0.60)


def glow_text(ax, *args, glow=3.2, **kwargs):
    txt = ax.text(*args, **kwargs)
    txt.set_path_effects([
        pe.withStroke(linewidth=glow, foreground=kwargs.get("color", gold)),
        pe.withStroke(linewidth=1.3, foreground=black)
    ])
    return txt


def draw_box(ax, xy, w, h, text, edge, fill=panel2, size=10):
    x, y = xy
    patch = FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.022,rounding_size=0.03",
                           linewidth=1.1, edgecolor=edge, facecolor=fill, alpha=0.90)
    ax.add_patch(patch)
    ax.text(x + w/2, y + h/2, text, ha="center", va="center", color=ice,
            fontsize=size, fontweight="bold",
            path_effects=[pe.withStroke(linewidth=1.8, foreground=black)])

# ============================================================
# INITIALIZATION OF THE OPERATOR PIPELINE
# ============================================================

In [ ]:
data     = DataEngine()
network  = NetworkEngine()
camera   = CameraEngine()
particles = ParticleEngine()
background = BackgroundEngine()

# Pre-compute systemic-state summaries before rendering.
# Reusing these arrays preserves a direct link between numerical states and frames.
cascade_size, max_contagion, avg_contagion = network.compute_cascade_metrics(FRAMES)

fig = plt.figure(figsize=FIGSIZE, facecolor=bg)
ax_bg = fig.add_axes([0, 0, 1, 1], zorder=-10)
ax_bg.set_facecolor(bg)
ax_bg.axis("off")

gs = fig.add_gridspec(3, 4, height_ratios=[0.16, 1.0, 0.34],
                      width_ratios=[1.05, 1.15, 1.15, 1.05],
                      hspace=0.17, wspace=0.15)

ax_title = fig.add_subplot(gs[0, :])
ax_data  = fig.add_subplot(gs[1, 0])
ax_net   = fig.add_subplot(gs[1, 1:3])
ax_model = fig.add_subplot(gs[1, 3])
ax_hud   = fig.add_subplot(gs[2, :])
axes = [ax_title, ax_data, ax_net, ax_model, ax_hud]

# ============================================================
# FRAME UPDATE - REALIZATION OF Psi_t FOLLOWED BY Pi
# ============================================================

In [ ]:
def update(frame):
    t = frame / max(FRAMES - 1, 1)

    if t < 0.22:
        scene, active = "DATA STREAM", cyan
    elif t < 0.48:
        scene, active = "RISK MAPPING", violet
    elif t < 0.76:
        scene, active = "NETWORK CONTAGION", red
    else:
        scene, active = "SYSTEMIC DYNAMICS", green

    # Global background intensity is a visual encoding of aggregate risk.
    intensity = 0.65 + 0.70 * data.mean_risk[frame]

    background.draw(ax_bg, frame, active, intensity)
    particles.draw(ax_bg, frame, active)

    for ax in axes:
        ax.clear()
        style_panel(ax)

    # ---------- TITLE ----------
    ax_title.axis("off")
    ax_title.set_facecolor(bg)
    glow_text(ax_title, 0.5, 0.62, "FINANCIAL INTELLIGENCE ENGINE",
              transform=ax_title.transAxes, ha="center", va="center",
              color=gold, fontsize=24, fontweight="bold", alpha=0.95,
              glow=3.8 + 1.4 * np.sin(frame * 0.16))
    ax_title.text(0.5, 0.18, "Operator-based Visual Analytics  •  Synthetic Systemic Risk",
                  transform=ax_title.transAxes, ha="center", va="center",
                  color=ice, fontsize=12, alpha=0.92)

    # ---------- DATA PANEL: latent and probabilistic state ----------
    # Psi_t maps p_i(t) into marker size; sampled Y_i controls display color.
    ax_data.set_title("LATENT RISK SPACE", color=gold, fontsize=12, pad=9)
    p_now = data.score_path[frame]
    colors = np.where(data.y == 1, red, cyan)
    sizes  = 16 + 58 * p_now
    ax_data.scatter(data.x1 + 0.035 * np.cos(frame * 0.045 + data.latent),
                    data.x2 + 0.035 * np.sin(frame * 0.045 + data.latent),
                    s=sizes, c=colors, alpha=0.64, edgecolors=white, linewidths=0.20)
    ax_data.set_xlim(-3.2, 3.2)
    ax_data.set_ylim(-3.2, 3.2)
    ax_data.grid(color=white, alpha=0.09)
    ax_data.tick_params(colors=steel, labelsize=8)
    ax_data.set_xlabel("$X_1$", color=steel, fontsize=9)
    ax_data.set_ylabel("$X_2$", color=steel, fontsize=9)
    ax_data.text(0.04, 0.93,
                 "cyan = stable\nred = crisis state\nsize ∝ risk score",
                 transform=ax_data.transAxes, color=ice, fontsize=8.2, va="top",
                 path_effects=[pe.withStroke(linewidth=1.8, foreground=black)])

    # ---------- NETWORK PANEL: graph state + contagion ----------
    # Pi projects coordinates; Psi_t then maps vulnerability/contagion to style.
    ax_net.axis("off")
    ax_net.set_title("SYSTEMIC RISK NETWORK  (graph distance contagion)", color=gold, fontsize=12, pad=9)
    ax_net.set_xlim(0, 1)
    ax_net.set_ylim(0, 1)

    pulse = 0.5 + 0.5 * np.sin(frame * 0.17)
    xp, yp, scale, alpha = camera.project(network.x, network.y, network.z, frame)

    # Edge encoding: appearance responds to contagion while retaining w_ij.
    edge_order = sorted(network.edges, key=lambda e: (network.z[e[0]] + network.z[e[1]]) / 2)
    for i, j, w in edge_order:
        c_level = max(network.contagion_level(i, t), network.contagion_level(j, t))
        col = red if c_level > 0.22 else steel
        lw  = 0.65 + 2.6 * w * c_level
        a   = 0.11 + 0.58 * c_level
        ax_net.plot([xp[i], xp[j]], [yp[i], yp[j]], color=col, alpha=a, lw=lw, zorder=1)

    # Shock rings are explanatory cues for S(t), not part of operator C.
    sx, sy = xp[network.source], yp[network.source]
    shock = np.clip((t - SHOCK_START) / SHOCK_DUR, 0, 1)
    if shock > 0.01:
        for rr in [0.10, 0.19, 0.29, 0.40]:
            radius = rr * shock
            if radius > 0.008:
                ax_net.add_patch(Circle((sx, sy), radius, fill=False, color=red,
                                        lw=1.1, alpha=max(0.01, 0.38 * (1 - rr) * shock), zorder=2))

    # Node encoding: vulnerability and contagion jointly determine appearance.
    order = np.argsort(network.z)
    for i in order:
        c_level = network.contagion_level(i, t)
        col = red if c_level > 0.30 else (green if network.risk[i] < 0.40 else cyan)
        node_size = (130 + 390 * network.risk[i] + 240 * c_level * pulse) * scale[i]
        node_alpha = np.clip(alpha[i] * (0.70 + 0.30 * c_level), 0.25, 1.0)

        ax_net.scatter(xp[i], yp[i], s=node_size * 1.45, color=col,
                       alpha=0.08 + 0.16 * c_level, linewidths=0, zorder=3)
        ax_net.scatter(xp[i], yp[i], s=node_size, color=col, alpha=node_alpha,
                       edgecolors=white, linewidths=0.65, zorder=4)
        ax_net.text(xp[i], yp[i], f"{i}", color=black, fontsize=6.5,
                    ha="center", va="center", fontweight="bold", zorder=5)

    draw_box(ax_net, (0.04, 0.81), 0.26, 0.10, "Shock source\n(max vulnerability)", red, size=9.5)
    draw_box(ax_net, (0.70, 0.81), 0.25, 0.10, "Perspective\nprojection", active, size=9.5)
    draw_box(ax_net, (0.35, 0.04), 0.30, 0.10, "Topology + vulnerability\n→ systemic exposure", gold, size=9.5)

    ax_net.text(0.5, 0.94, scene, color=active, fontsize=12.5, ha="center", fontweight="bold",
                path_effects=[pe.withStroke(linewidth=2.3, foreground=black)])

    # ---------- SYSTEMIC METRICS PANEL ----------
    # Probabilistic risk, cascade extent, and peak contagion remain distinct.
    ax_model.set_title("SYSTEMIC METRICS", color=gold, fontsize=12, pad=9)
    xx = np.arange(frame + 1)
    ax_model.plot(xx, data.mean_risk[:frame+1],     color=cyan,  lw=2.1, label="Mean Risk")
    ax_model.plot(xx, cascade_size[:frame+1] / N_NODES, color=red, lw=2.0, label="Cascade Size (norm)")
    ax_model.plot(xx, max_contagion[:frame+1],      color=green, lw=1.9, label="Max Contagion")

    ax_model.set_xlim(0, FRAMES)
    ax_model.set_ylim(0, 1.05)
    ax_model.grid(color=white, alpha=0.10)
    ax_model.tick_params(colors=steel, labelsize=8)
    ax_model.legend(facecolor=panel, edgecolor=gold, labelcolor=ice, fontsize=7.5, loc="lower right")

    ax_model.text(0.05, 0.82,
                  f"Mean Risk  = {data.mean_risk[frame]:.3f}\n"
                  f"Cascade    = {int(cascade_size[frame])}/{N_NODES}\n"
                  f"Max Contag = {max_contagion[frame]:.3f}",
                  transform=ax_model.transAxes, color=ice, fontsize=10.5,
                  family="monospace",
                  path_effects=[pe.withStroke(linewidth=1.8, foreground=black)])

    # ---------- SCIENTIFIC HUD ----------
    # Displays governing relationships and the operator sequence.
    ax_hud.axis("off")
    ax_hud.text(0.03, 0.70, r"Risk signal:  $p_i(t) = \sigma(z_i + \varepsilon_i(t))$",
                color=ice, fontsize=13, family="monospace")
    ax_hud.text(0.03, 0.42, r"Pipeline:  latent map  →  probabilistic scores  →  graph contagion  →  visualization",
                color=gold, fontsize=13, family="monospace",
                path_effects=[pe.withStroke(linewidth=1.8, foreground=black)])
    ax_hud.text(0.03, 0.16, "Contagion uses shortest-path distance on the financial network.",
                color=steel, fontsize=11.5, family="monospace")

    ax_hud.text(0.72, 0.72, f"SCENE: {scene}\nFRAME: {frame+1:03d}/{FRAMES}",
                color=ice, fontsize=11, family="monospace")
    ax_hud.add_patch(Rectangle((0.72, 0.40), 0.22, 0.06, fill=False, edgecolor=steel, linewidth=1.0))
    ax_hud.add_patch(Rectangle((0.72, 0.40), 0.22 * (frame + 1) / FRAMES, 0.06, color=active, alpha=0.85))

    ax_hud.text(0.50, 0.02, "@IsabelCasPe  •  Scientific Visualization",
                color=gold, fontsize=13, family="monospace", ha="center",
                path_effects=[pe.withStroke(linewidth=1.8, foreground=black)])
    ax_bg.text(
        0.50, 0.50, "© @IsabelCasPe :)",
        transform=ax_bg.transAxes,
        ha="center", va="center",
        fontsize=50,
        color=white,
        alpha=0.022,
        fontweight="bold",
        zorder=-8
    )

    fig.patches.clear()
    fig.patches.append(Rectangle((0.011, 0.020), 0.978, 0.948, transform=fig.transFigure,
                                 fill=False, edgecolor=gold, linewidth=1.25, alpha=0.70))

# ============================================================
# EXECUTION + REPRODUCIBILITY OUTPUTS
# ============================================================

In [ ]:
ani = FuncAnimation(fig, update, frames=FRAMES, interval=1000 / FPS, repeat=False)

writer = FFMpegWriter(fps=FPS, bitrate=6500)
ani.save(OUT_VIDEO, writer=writer, dpi=DPI)
print(f"Video saved: {OUT_VIDEO}")

# Persist principal numerical states so the synthetic experiment can be
# audited independently of the animation rendering.
np.savez(
    OUT_DATA,
    score_path=data.score_path,
    mean_risk=data.mean_risk,
    cascade_size=cascade_size,
    max_contagion=max_contagion,
    avg_contagion=avg_contagion,
    node_risk=network.risk,
    source=network.source,
    distances=dict(network.distances),
    edges=network.edges,
    x1=data.x1,
    x2=data.x2,
    latent=data.latent
)
print(f"Scientific results saved: {OUT_DATA}")
print("Done.")